In [2]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Loading model...")
model = joblib.load('master_svm_model.pkl')

def normalize_landmarks(landmarks_63):
    pts = np.array(landmarks_63).reshape(21, 3)
    pts = pts - pts[0]
    max_val = np.max(np.abs(pts))
    if max_val > 0: pts = pts / max_val
    return pts.flatten()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
print("Webcam started. Press 'q' to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
        
    frame = cv2.flip(frame, 1)
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            raw_landmarks = []
            for lm in hand_landmarks.landmark:
                raw_landmarks.extend([lm.x, lm.y, lm.z])
            normalized = normalize_landmarks(raw_landmarks)
            
            landmarks_array = np.array(normalized).reshape(1, -1)
            prediction = model.predict(landmarks_array)[0]
            
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            cv2.putText(frame, f"Prediction: {prediction}", (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
    else:
        cv2.putText(frame, "No Hand Detected", (10, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
    cv2.imshow('Live ASL Test', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

Loading model...
Webcam started. Press 'q' to quit.


In [3]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import warnings
from collections import deque, Counter

warnings.filterwarnings('ignore')

print("Loading model...")
model = joblib.load('master_svm_model.pkl')

def normalize_landmarks(landmarks_63):
    pts = np.array(landmarks_63).reshape(21, 3)
    pts = pts - pts[0]
    max_val = np.max(np.abs(pts))
    if max_val > 0: pts = pts / max_val
    return pts.flatten()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 10-frame smoothing buffer to stop the flickering
prediction_buffer = deque(maxlen=10)

cap = cv2.VideoCapture(0)
print("Webcam started. Press 'q' to quit.")
print("Hold your hand steady for a second so the buffer can fill up!")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
        
    frame = cv2.flip(frame, 1)
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            raw_landmarks = []
            for lm in hand_landmarks.landmark:
                raw_landmarks.extend([lm.x, lm.y, lm.z])
            normalized = normalize_landmarks(raw_landmarks)
            
            landmarks_array = np.array(normalized).reshape(1, -1)
            
            # Force the prediction (No confidence threshold!)
            prediction = model.predict(landmarks_array)[0]
            probabilities = model.predict_proba(landmarks_array)[0]
            confidence = max(probabilities) * 100
            
            # Add to smoothing buffer
            prediction_buffer.append(prediction)
                
            if len(prediction_buffer) > 0:
                most_common = Counter(prediction_buffer).most_common(1)[0][0]
            else:
                most_common = "..."
            
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Display the smoothed letter and the raw confidence
            cv2.putText(frame, f"Letter: {most_common}", (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
            cv2.putText(frame, f"Confidence: {confidence:.1f}%", (10, 90), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            
    else:
        prediction_buffer.clear()
        cv2.putText(frame, "No Hand Detected", (10, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
    cv2.imshow('Live ASL Test', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

Loading model...
Webcam started. Press 'q' to quit.
Hold your hand steady for a second so the buffer can fill up!
